In [2]:
import pandas as pd
import numpy as np
from tools import sherlock

In [3]:
df = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])

In [4]:
sherlock(df)

1394 lignes | 9 colonnes | 0 lignes doublons
--------------------------------------------------
                            Type  Manquants Manquants %  Uniques
code_journal                 str          0        0.0%        5
date_écriture     datetime64[us]          0        0.0%      297
code_compte                  str          0        0.0%       73
intitulé_compte              str          0        0.0%       74
pièce                        str        185      13.27%      613
libellé_écriture             str          0        0.0%      348
débit_euro               float64          0        0.0%      479
crédit_euro              float64          0        0.0%      454
année                      int64          0        0.0%        2
--------------------------------------------------
Pas de colonne éligible en PK


In [5]:
df.head()

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
0,AC,2024-11-05,FMER,AXESCIBLES MER,2411-212,Honoraire AXESCIBLES MER,0.0,960.0,2024
1,AC,2024-11-05,62260000,Honoraires,2411-212,Honoraire AXESCIBLES MER,800.0,0.0,2024
2,AC,2024-11-05,62260000,Honoraires,2411-212,Honoraire AXESCIBLES MER,160.0,0.0,2024
3,ACH,2024-05-23,10810000,Compte de l'exploitant,2405-ACH -000001,POINT P P,0.0,54.0,2024
4,ACH,2024-05-23,21810000,"Installations générales, agencements",2405-ACH -000001,POINT P P,54.0,0.0,2024


---
# CA

code_compte == '70600000' -> loyers

## CA Global

In [6]:
loyers = df.query("code_compte == '70600000'")

In [7]:
credit = loyers['crédit_euro'].sum()
debit = loyers['débit_euro'].sum()
ca_global = credit - debit
print(f"Chiffre d'affaire = {ca_global:.2f} €")

Chiffre d'affaire = 26945.37 €


## CA Annuel

In [8]:
ca_annuel = loyers.groupby('année')[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_annuel

,année,crédit_euro,débit_euro,ca_net
0,2024,9582.69,0.00,9582.69
1,2025,17976.49,613.81,17362.68


## CA Mensuel

In [9]:
ca_mensuel = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_mensuel

,date_écriture,crédit_euro,débit_euro,ca_net
0,2024-07,1037.27,0.00,1037.27
1,2024-08,3325.04,0.00,3325.04
2,2024-09,1183.40,0.00,1183.40
3,2024-10,1332.25,0.00,1332.25
4,2024-11,1038.85,0.00,1038.85
5,2024-12,1665.88,0.00,1665.88
6,2025-01,2095.50,613.81,1481.69
7,2025-02,994.52,0.00,994.52
8,2025-03,955.33,0.00,955.33
9,2025-04,1910.08,0.00,1910.08


## CA par Plateforme

In [10]:
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

### CA par PTF

In [11]:
ca_ptf_y = loyers.groupby('canal')[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_ptf_y['%'] = ca_ptf_y['ca_net'] / ca_global * 100
ca_ptf_y

,canal,crédit_euro,débit_euro,ca_net,%
0,AIRBNB,22066.90,310.41,21756.49,80.742963
1,BOOKING,4717.64,0.00,4717.64,17.508166
2,DIRECT,774.64,303.40,471.24,1.748872


### CA par PTF annuel

In [12]:
loyers.groupby(['année', 'canal'])[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()

,année,canal,crédit_euro,débit_euro,ca_net
0,2024,AIRBNB,8383.79,0.00,8383.79
1,2024,BOOKING,1148.90,0.00,1148.90
2,2024,DIRECT,50.00,0.00,50.00
3,2025,AIRBNB,13683.11,310.41,13372.70
4,2025,BOOKING,3568.74,0.00,3568.74
5,2025,DIRECT,724.64,303.40,421.24


### CA par PTF mensuel

In [13]:
loyers.groupby([loyers['date_écriture'].dt.to_period('M'), 'canal'])[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()

,date_écriture,canal,crédit_euro,débit_euro,ca_net
0,2024-07,AIRBNB,1037.27,0.00,1037.27
1,2024-08,AIRBNB,3005.59,0.00,3005.59
2,2024-08,BOOKING,319.45,0.00,319.45
3,2024-09,AIRBNB,1122.10,0.00,1122.10
4,2024-09,BOOKING,61.30,0.00,61.30
5,2024-10,AIRBNB,1332.25,0.00,1332.25
6,2024-11,AIRBNB,308.48,0.00,308.48
7,2024-11,BOOKING,680.37,0.00,680.37
8,2024-11,DIRECT,50.00,0.00,50.00
9,2024-12,AIRBNB,1578.10,0.00,1578.10


---
# Charges

In [14]:
charges = df[df['code_compte'].str.startswith('6') & (df['débit_euro'] > 0)]

## Charges annuelles

In [15]:
charges.groupby('année')['débit_euro'].sum().reset_index()

,année,débit_euro
0,2024,24391.63
1,2025,13245.40


## Details par Categories

In [16]:
charges.groupby(['code_compte', 'intitulé_compte'])['débit_euro'].sum().reset_index()

,code_compte,intitulé_compte,débit_euro
0,60611000,Eau,1190.63
1,60614000,Carburant,55.80
2,60630000,Entretien et petit équipement,10434.79
3,61520000,Entretien sur biens immobiliers,1188.72
4,61560000,Maintenance,179.90
5,61610000,Primes d'assurance,1989.88
6,62220000,Commissions sur ventes,1552.39
7,62260000,Honoraires,1919.96
8,62270000,Frais d'actes et de contentieux,5945.49
9,62310000,Publicité,365.80


### Focus sur '60611000' : Eau

In [17]:
charges.query("code_compte == '60611000'").sort_values('débit_euro', ascending=False).head(10)

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
978,ACH,2025-09-23,60611000,Eau,23092025,SYNDIC EAUX 01/07/24-30/06/25,1023.57,0.0,2025
977,ACH,2025-04-07,60611000,Eau,07042025,SYNDIC EAU ACPTE 01/07/24-30/06/25,62.30,0.0,2025
204,ACH,2024-05-23,60611000,Eau,2405-ACH -000100,SYNDICAT DES EAUX 01/07/23 - 30/06/24,50.00,0.0,2024
388,ACH,2024-09-20,60611000,Eau,2409-ACH -000013,EAU 22/2-30/6,36.02,0.0,2024
206,ACH,2024-05-23,60611000,Eau,026873,SYNDICAT DES EAUX 01/07/23 - 30/06/24,18.74,0.0,2024


### Focus sur '60630000' : Entretien et petit équipement

In [18]:
charges.query("code_compte == '60630000'").sort_values('débit_euro', ascending=False).head(10)

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
46,ACH,2024-05-23,60630000,Entretien et petit équipement,4179-40,MAXXILOT,718.79,0.0,2024
274,ACH,2024-06-05,60630000,Entretien et petit équipement,410409,AUCHAN Ordi et imprimante Lodge paye perso,599.98,0.0,2024
1000,ACH,2025-04-17,60630000,Entretien et petit équipement,01/0035234,CONFORAMA sèche linge,379.99,0.0,2025
22,ACH,2024-05-23,60630000,Entretien et petit équipement,2405-ACH -000010,MAXXILOT,377.80,0.0,2024
446,ACH,2024-10-15,60630000,Entretien et petit équipement,950613,CONFO Payé Cb Lodge,359.99,0.0,2024
1001,ACH,2025-04-21,60630000,Entretien et petit équipement,1248845,BUT surmatelas,321.97,0.0,2025
52,ACH,2024-05-23,60630000,Entretien et petit équipement,0.008,CONFORAMA,299.00,0.0,2024
64,ACH,2024-05-23,60630000,Entretien et petit équipement,938456,CONFORAMA,298.59,0.0,2024
228,ACH,2024-05-23,60630000,Entretien et petit équipement,938456,CONFORAMA Four pyrolyse Lodge payé CB perso theo,288.59,0.0,2024
164,ACH,2024-05-23,60630000,Entretien et petit équipement,1433456731,IKEA Payé perso,252.86,0.0,2024


### Focus sur '66116000' : Charges d'intérêts sur emprunt

In [19]:
charges.query("code_compte == '66116000'").sort_values('débit_euro', ascending=False).head(10)

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
694,OD,2024-12-31,66116000,Charges d'intérêts sur emprunt,2412-OD -000001,Rbt prêt 2024,2917.94,0.0,2024
1132,OD,2025-01-05,66116000,Charges d'intérêts sur emprunt,2501-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,306.51,0.0,2025
1133,OD,2025-02-05,66116000,Charges d'intérêts sur emprunt,2502-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,305.34,0.0,2025
1134,OD,2025-03-05,66116000,Charges d'intérêts sur emprunt,2503-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,304.17,0.0,2025
1135,OD,2025-04-05,66116000,Charges d'intérêts sur emprunt,2504-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,302.99,0.0,2025
1136,OD,2025-05-05,66116000,Charges d'intérêts sur emprunt,2505-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,301.80,0.0,2025
1137,OD,2025-06-05,66116000,Charges d'intérêts sur emprunt,2506-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,300.61,0.0,2025
1138,OD,2025-07-05,66116000,Charges d'intérêts sur emprunt,2507-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,299.41,0.0,2025
1139,OD,2025-08-05,66116000,Charges d'intérêts sur emprunt,2508-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,298.21,0.0,2025
1140,OD,2025-09-05,66116000,Charges d'intérêts sur emprunt,2509-REMP-000001,Intérêts des emprunts H1721385/9973466/813545E,297.00,0.0,2025


### Focus sur '62270000' : Frais d'actes et de contentieux

In [20]:
charges.query("code_compte == '62270000'").sort_values('débit_euro', ascending=False).head(10)

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
678,OD,2024-05-23,62270000,Frais d'actes et de contentieux,2405-OD -000001,Achat maison frais d'acte,3526.22,0.0,2024
679,OD,2024-05-23,62270000,Frais d'actes et de contentieux,2405-OD -000001,Achat maison frais de notaire,2419.27,0.0,2024


### Focus sur '62510000' : Voyages et déplacements

In [21]:
charges.query("code_compte == '62510000'").sort_values('débit_euro', ascending=False).head(10)

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
700,OD,2024-12-31,62510000,Voyages et déplacements,2412-OD -000004,ik 2024,3464.0,0.0,2024


### Focus sur '62220000' : Commissions sur ventes

In [22]:
comm = charges.query("code_compte == '62220000'")
comm.groupby('année')['débit_euro'].sum().reset_index()

,année,débit_euro
0,2024,486.38
1,2025,1066.01


In [23]:
conds = [
    comm['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    comm['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
comm['canal'] = np.select(conds, choices, default='DIRECT')
comm_y = comm.groupby('canal')['débit_euro'].sum().reset_index()
comm_y

,canal,débit_euro
0,AIRBNB,781.16
1,BOOKING,771.23


Taux de commissions par plateforme

In [24]:
df_comm = ca_ptf_y.merge(comm_y, how='inner', on='canal')
df_comm['tx_comm'] = df_comm['débit_euro_y'] / df_comm['ca_net'] * 100
df_comm

,canal,crédit_euro,débit_euro_x,ca_net,%,débit_euro_y,tx_comm
0,AIRBNB,22066.90,310.41,21756.49,80.742963,781.16,3.590469
1,BOOKING,4717.64,0.00,4717.64,17.508166,771.23,16.347793


### Focus sur '61610000' : Primes d'assurance

In [25]:
charges.query("code_compte == '61610000'").groupby('année')['débit_euro'].sum().reset_index()

,année,débit_euro
0,2024,262.96
1,2025,1726.92


## Charges Mensuelles

In [26]:
charges.groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()

,date_écriture,débit_euro
0,2024-05,10109.31
1,2024-06,1259.58
2,2024-07,902.24
3,2024-08,882.86
4,2024-09,640.87
5,2024-10,1159.22
6,2024-11,1308.93
7,2024-12,8128.62
8,2025-01,877.07
9,2025-02,1667.56


### Focus sur 12/2025

In [27]:
charges[(charges['date_écriture'].dt.to_period('M') == '2024-12')]

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
478,ACH,2024-12-20,60630000,Entretien et petit équipement,02400,LECLERC,52.87,0.0,2024
480,ACH,2024-12-31,62310000,Publicité,GCFRD0007924008,GOOGLE Payé CB Sarah perso chaque mois,13.34,0.0,2024
482,ACH,2024-12-30,60630000,Entretien et petit équipement,1236501,BUT,89.99,0.0,2024
484,ACH,2024-12-18,62620000,Frais de télécommunication,02C242G418 24J2-,ORANGE PLVMT compte commun perso,25.99,0.0,2024
658,BQ,2024-12-05,61610000,Primes d'assurance,4096826,VIR SEPA M MOIZAN BERNARD OU M THELEM ASS LODG...,32.87,0.0,2024
660,BQ,2024-12-10,62780000,Frais bancaires,4811805,* COTISATION FREELANCE INITIAL,9.90,0.0,2024
675,BQ,2024-12-31,61610000,Primes d'assurance,4737766,VIR SEPA M MOIZAN BERNARD OU M THELEM ASS LODG...,32.87,0.0,2024
694,OD,2024-12-31,66116000,Charges d'intérêts sur emprunt,2412-OD -000001,Rbt prêt 2024,2917.94,0.0,2024
696,OD,2024-12-31,62220000,Commissions sur ventes,2412-OD -000002,Com airbnb,299.74,0.0,2024
700,OD,2024-12-31,62510000,Voyages et déplacements,2412-OD -000004,ik 2024,3464.00,0.0,2024


## Mensuel de la Categorie "Entretien et petit équipement"

In [28]:
charges.query("code_compte == '60630000'").groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()

,date_écriture,débit_euro
0,2024-05,3888.96
1,2024-06,1027.27
2,2024-07,516.38
3,2024-08,578.90
4,2024-09,388.13
5,2024-10,851.54
6,2024-11,121.07
7,2024-12,142.86
8,2025-01,219.35
9,2025-02,173.00


### Focus sur 04/2025

In [29]:
charges[(charges['code_compte'] == '60630000') & (charges['date_écriture'].dt.to_period('M') == '2025-04')]

,code_journal,date_écriture,code_compte,intitulé_compte,pièce,libellé_écriture,débit_euro,crédit_euro,année
994,ACH,2025-04-08,60630000,Entretien et petit équipement,500044586,TRUFFAUT,7.98,0.0,2025
995,ACH,2025-04-08,60630000,Entretien et petit équipement,010004346,BOUCHARA,16.99,0.0,2025
996,ACH,2025-04-08,60630000,Entretien et petit équipement,1872500007139,LEROY MERLIN,6.17,0.0,2025
997,ACH,2025-04-13,60630000,Entretien et petit équipement,51030702038,INTERMARCHE,28.13,0.0,2025
998,ACH,2025-04-15,60630000,Entretien et petit équipement,500044585,TRUFFAUT,21.98,0.0,2025
999,ACH,2025-04-15,60630000,Entretien et petit équipement,49700,AUCHAN,126.42,0.0,2025
1000,ACH,2025-04-17,60630000,Entretien et petit équipement,01/0035234,CONFORAMA sèche linge,379.99,0.0,2025
1001,ACH,2025-04-21,60630000,Entretien et petit équipement,1248845,BUT surmatelas,321.97,0.0,2025
1002,ACH,2025-04-21,60630000,Entretien et petit équipement,25-04-37,L'EPICERIE CAFE jus de fruits,88.80,0.0,2025
1003,ACH,2025-04-28,60630000,Entretien et petit équipement,412000686401000,BRICOMARCHE,50.19,0.0,2025


## Fixes vs Variables

In [30]:
fixes = ['61610000', '66116000', '62620000', '62780000', '63512000', '62810000', '68112000']
semi_fixes = ['60630000', '61520000', '61560000']
variables = ['60611000', '60614000', '62220000', '62260000', '62270000', '62310000', '62340000', '62510000', '62610000', '65800000']

conds = [
    charges['code_compte'].isin(fixes),
    charges['code_compte'].isin(semi_fixes),
    charges['code_compte'].isin(variables)
]

choices = ['fixe', 'semi_fixe', 'variable']

charges['categorie'] = np.select(conds, choices, default='autre')

In [31]:
# Sommes par Catégories

charges.groupby('categorie')['débit_euro'].sum()

categorie
fixe         11308.46
semi_fixe    11803.41
variable     14525.16
Name: débit_euro, dtype: float64

---
# Cashflow

## Mensuel

In [32]:
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']
cashflow

,date_écriture,ca_net,charges,cashflow
0,2024-05,0.00,10109.31,-10109.31
1,2024-06,0.00,1259.58,-1259.58
2,2024-07,1037.27,902.24,135.03
3,2024-08,3325.04,882.86,2442.18
4,2024-09,1183.40,640.87,542.53
5,2024-10,1332.25,1159.22,173.03
6,2024-11,1038.85,1308.93,-270.08
7,2024-12,1665.88,8128.62,-6462.74
8,2025-01,1481.69,877.07,604.62
9,2025-02,994.52,1667.56,-673.04


## Annuel

In [33]:
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('Y'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('Y'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']
cashflow

,date_écriture,ca_net,charges,cashflow
0,2024,9582.69,24391.63,-14808.94
1,2025,17362.68,13245.40,4117.28


---
---
# Correction anomalie juillet 2025

Juillet 2025 présente une chute anormale documentée (-82% consultations Airbnb, -94% réservations vs juillet 2024). Le CA comptable brut de 17 362€ est donc sous-estimé. On corrige en remplaçant le CA juillet 2025 réel par la moyenne des mois adjacents (avril à septembre 2025), plus représentative d'un juillet normal.

In [34]:
ca_mensuel

,date_écriture,crédit_euro,débit_euro,ca_net
0,2024-07,1037.27,0.00,1037.27
1,2024-08,3325.04,0.00,3325.04
2,2024-09,1183.40,0.00,1183.40
3,2024-10,1332.25,0.00,1332.25
4,2024-11,1038.85,0.00,1038.85
5,2024-12,1665.88,0.00,1665.88
6,2025-01,2095.50,613.81,1481.69
7,2025-02,994.52,0.00,994.52
8,2025-03,955.33,0.00,955.33
9,2025-04,1910.08,0.00,1910.08


In [35]:
mois_ref = ['2025-04', '2025-05', '2025-06', '2025-08', '2025-09']
mask = ca_mensuel['date_écriture'].astype(str).isin(mois_ref)

ca_cible = ca_mensuel[mask]['ca_net'].mean()
ca_juil25 = ca_mensuel[ca_mensuel['date_écriture'].astype(str) == '2025-07']['ca_net'].item()
manque = ca_cible - ca_juil25

ca2025_corrige = ca_annuel.query("année == 2025")['ca_net'].item() + manque

print(f"CA Juillet 2025 = {ca_juil25:.2f} €")
print(f"Moyenne saison haute = {ca_cible:.2f} €")
print(f"Manque à gagner = {manque:.2f} €")
print(f"CA 2025 corrigé = {ca2025_corrige:.2f} €")

CA Juillet 2025 = 718.18 €
Moyenne saison haute = 1981.46 €
Manque à gagner = 1263.28 €
CA 2025 corrigé = 18625.96 €


---
# RevPAR 2025 — Estimation comptable

Le RevPAR (Revenue Per Available Room/Night) mesure le revenu généré par jour disponible, qu'il soit vendu ou non. Contrairement à l'ADR (prix moyen par nuit vendue), il intègre le taux d'occupation — c'est l'indicateur de performance globale d'un hébergement.

Le RevPAR 2024 réel a été calculé à partir des données de taxe de séjour (nuits occupées + jours disponibles réels par mois) → voir notebook 7.

Pour 2025, la taxe de séjour n'est pas disponible. Le RevPAR est donc estimé à partir du CA comptable annuel divisé par une hypothèse de 340 jours disponibles (25 jours réservés maintenance/usage propriétaires). C'est une approximation annuelle — le détail mensuel n'est pas fiable à cause du décalage entre date de séjour et date d'encaissement.

In [36]:
ca2025 = ca_annuel.query("année == 2025")['ca_net']
revpar2025 = ca2025.item() / 340
revpar2025_corrige = ca2025_corrige / 340
print(f"RevPAR 2025 = {revpar2025:.2f} €")
print(f"RevPAR 2025 (sur CA corrigé) = {revpar2025_corrige:.2f} €")

RevPAR 2025 = 51.07 €
RevPAR 2025 (sur CA corrigé) = 54.78 €
